In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/raw/customers.csv')

# Fix TotalCharges and Churn (same as EDA)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(0, inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Loaded:", df.shape)

Loaded: (7043, 21)


In [3]:
# customerID is just an identifier, not useful for modeling
df.drop(columns=['customerID'], inplace=True)

print("Columns remaining:", df.shape[1])

Columns remaining: 20


In [4]:
# 1. Charge per month relative to tenure (high charge + low tenure = risky)
df['charge_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

# 2. Has multiple services (bundled customers churn less)
services = ['PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']
df['total_services'] = (df[services] == 'Yes').sum(axis=1)

# 3. Is a new customer (tenure <= 12 months)
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)

# 4. Is high value (monthly charges above median)
df['is_high_value'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

# 5. Has no support services (vulnerable to churn)
df['no_support_services'] = (
    (df['OnlineSecurity'] == 'No') & 
    (df['TechSupport'] == 'No')
).astype(int)

print("New features created!")
print(df[['charge_per_tenure', 'total_services', 
          'is_new_customer', 'is_high_value', 
          'no_support_services']].head())

New features created!
   charge_per_tenure  total_services  is_new_customer  is_high_value  \
0          14.925000               1                1              0   
1           1.627143               3                0              0   
2          17.950000               3                1              0   
3           0.919565               3                0              0   
4          23.566667               1                1              1   

   no_support_services  
0                    1  
1                    0  
2                    0  
3                    0  
4                    1  


In [5]:
# Yes/No columns → 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Gender → 1/0
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

print("Binary encoding done!")

Binary encoding done!


In [6]:
# These columns have 3+ categories → use one-hot encoding
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
              'OnlineBackup', 'DeviceProtection', 'TechSupport',
              'StreamingTV', 'StreamingMovies', 'Contract',
              'PaymentMethod']

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print("Shape after encoding:", df.shape)
print("Columns:", df.columns.tolist())

Shape after encoding: (7043, 36)
Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'charge_per_tenure', 'total_services', 'is_new_customer', 'is_high_value', 'no_support_services', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [7]:
scaler = MinMaxScaler()

scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 
              'charge_per_tenure', 'total_services']

df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("Scaling done!")
print(df[scale_cols].describe().round(2))

Scaling done!
        tenure  MonthlyCharges  TotalCharges  charge_per_tenure  \
count  7043.00         7043.00       7043.00            7043.00   
mean      0.45            0.46          0.26               0.07   
std       0.34            0.30          0.26               0.11   
min       0.00            0.00          0.00               0.00   
25%       0.12            0.17          0.05               0.01   
50%       0.40            0.52          0.16               0.02   
75%       0.76            0.71          0.44               0.07   
max       1.00            1.00          1.00               1.00   

       total_services  
count         7043.00  
mean             0.42  
std              0.26  
min              0.00  
25%              0.12  
50%              0.38  
75%              0.62  
max              1.00  


In [9]:
print("=" * 40)
print("  FEATURE ENGINEERING SUMMARY")
print("=" * 40)
print(f"\n Total rows    : {df.shape[0]}")
print(f" Total features: {df.shape[1]}")
print(f" Churn rate    : {df['Churn'].mean()*100:.1f}%")
print(f" Missing values: {df.isnull().sum().sum()}")
print("\n Sample:")
print(df.head(3))

  FEATURE ENGINEERING SUMMARY

 Total rows    : 7043
 Total features: 36
 Churn rate    : 26.5%
 Missing values: 0

 Sample:
   gender  SeniorCitizen  Partner  Dependents    tenure  PhoneService  \
0       0              0        1           0  0.013889             0   
1       1              0        0           0  0.472222             1   
2       1              0        0           0  0.027778             1   

   PaperlessBilling  MonthlyCharges  TotalCharges  Churn  ...  \
0                 1        0.115423      0.003437      0  ...   
1                 0        0.385075      0.217564      0  ...   
2                 1        0.354229      0.012453      1  ...   

   TechSupport_Yes  StreamingTV_No internet service  StreamingTV_Yes  \
0            False                            False            False   
1            False                            False            False   
2            False                            False            False   

   StreamingMovies_No internet s

In [10]:
df.to_csv('../data/processed/features.csv', index=False)

print("Saved to data/processed/features.csv ✅")

Saved to data/processed/features.csv ✅
